# Text Watermarking Lab — from Bacon's cipher to keyed randomness

Companion notebook for the ilm.red post *AI text watermarking: how keyed randomness marks machine text*.
Everything here is **published-method detection and education**: you will encode a Bacon biliteral
message, build a toy green-list watermark (Kirchenbauer et al. 2023, arXiv:2301.10226), detect it
with a z-score, and see why detection confidence grows with length and vanishes on short or
constrained text — the same limits Anthropic states for Claude's SynthID-Text-based watermark.

No GPU, no API keys, standard library only. For the production-grade implementation, see Google's
open-sourced [`synthid-text`](https://github.com/google-deepmind/synthid-text) (Nature 2024).


## 1 · Bacon's biliteral cipher (1605 / 1623)

Francis Bacon hid messages in the *pattern of choices* between two barely-different letterforms
while the visible text reads normally — the signal lives in **which of two equally good choices**
is made. Four centuries later, that is exactly where an LLM watermark lives. Here the two
'letterforms' are lowercase (a-form) and UPPERCASE (b-form) of an innocent cover text.


In [ ]:
BACON = {c: format(i, '05b') for i, c in enumerate('ABCDEFGHIKLMNOPQRSTUWXYZ')}  # 24-letter alphabet (I=J, U=V)
INV = {v: k for k, v in BACON.items()}

def encode(secret, cover):
    bits = ''.join(BACON[c] for c in secret.upper().replace('J','I').replace('V','U') if c in BACON)
    out, i = [], 0
    for ch in cover:
        if ch.isalpha() and i < len(bits):
            out.append(ch.upper() if bits[i] == '1' else ch.lower()); i += 1
        else:
            out.append(ch)
    assert i == len(bits), 'cover text too short'
    return ''.join(out)

def decode(stego):
    bits = ''.join('1' if ch.isupper() else '0' for ch in stego if ch.isalpha())
    return ''.join(INV.get(bits[i:i+5], '?') for i in range(0, len(bits) - len(bits) % 5, 5))

cover = 'the quiet art of hiding a signal inside ordinary choices was known long before computers existed here'
stego = encode('WATERMARK', cover)
print(stego)
print('decoded:', decode(stego).rstrip('A'))  # letters after the message are unmodulated lowercase = null padding, as in Bacon's own practice


The reader sees odd capitalisation; Bacon's real cipher used two typefaces so subtle the reader saw
nothing at all. The *key* is knowing the encoding exists and how to read it — precisely the role of
the watermark key in SynthID-style schemes.


## 2 · A toy green-list watermark (Kirchenbauer et al., 2023)

The **green-list family**: at each step, a hash of the previous token seeds a split of the
vocabulary into a *green* and *red* half, and generation softly boosts green tokens. Detection
counts green hits — no model needed, only the key. (Note: this family *re-weights* the
distribution and is therefore distortionary in principle; the **sampling-source family** used by
SynthID/Claude changes only the randomness source. Keep the two distinct — most explainers don't.)


In [ ]:
import hashlib, random

WORDS = ('the of and a to in is it that was for on are as with his they at be this from I have '
         'or by one had not but what all were when we there can an your which their said if do '
         'will each about how up out them then she many some so these would other into more her '
         'two like him see time could no make than first been its who now people my made over').split()

def green_set(prev_word, key, vocab=WORDS):
    h = hashlib.sha256(f'{key}:{prev_word}'.encode()).digest()
    rng = random.Random(h)
    shuffled = sorted(vocab, key=lambda w: rng.random())
    return set(shuffled[:len(vocab)//2])

def generate(n, key=None, delta=3.0, seed=0):
    'A zero-knowledge toy LM: uniform over WORDS, optionally green-boosted. Structure-free on purpose.'
    rng = random.Random(seed)
    out = ['the']
    for _ in range(n):
        weights = []
        greens = green_set(out[-1], key) if key is not None else set()
        for w in WORDS:
            weights.append(delta if w in greens else 1.0)
        out.append(rng.choices(WORDS, weights=weights)[0])
    return out[1:]


In [ ]:
import statistics

def z_score(tokens, key):
    'Fraction of tokens in the keyed green set vs the 1/2 expected by chance.'
    hits, prev = 0, 'the'
    for w in tokens:
        if w in green_set(prev, key): hits += 1
        prev = w
    n = len(tokens)
    return (hits - n/2) / (n * 0.25) ** 0.5   # binomial z under H0: p = 1/2

KEY = 'ilm-red-demo-key'
marked   = generate(400, key=KEY)
unmarked = generate(400, key=None)
print(f'watermarked   z = {z_score(marked, KEY):6.2f}   (way past any significance line)')
print(f'unwatermarked z = {z_score(unmarked, KEY):6.2f}   (chance)')
print(f'wrong key     z = {z_score(marked, "someone-elses-key"):6.2f}   (a different provider sees nothing)')


## 3 · Why length is everything

Detection is a statistical claim, and short text simply carries too few keyed choices. Watch the
z-score as passage length grows — and remember the announced limits: sparse on factual text,
exact code, and light proofreading, where the model has few *free* choices at all.


In [ ]:
for n in (10, 25, 50, 100, 200, 400, 800):
    zs = [z_score(generate(n, key=KEY, seed=s), KEY) for s in range(20)]
    print(f'n={n:4d}   mean z = {statistics.mean(zs):5.2f}   min = {min(zs):5.2f}')


## 4 · Index of coincidence — the keyless detector's world

Without the key you are back to *statistical tells* (what Pangram-style detectors do). The index
of coincidence is the classical cryptanalyst's version: it measures how far letter frequencies
drift from uniform. It can tell English from noise — it cannot tell watermarked English from
unwatermarked, which is the whole asymmetry between key-holders and everyone else.


In [ ]:
def ioc(text):
    s = [c for c in text.lower() if c.isalpha()]
    n = len(s)
    freqs = {c: s.count(c) for c in set(s)}
    return sum(f*(f-1) for f in freqs.values()) / (n*(n-1)) if n > 1 else 0

english = 'the quiet art of hiding a signal inside ordinary choices was known long before computers'
import random as _r; noise = ''.join(_r.choice('abcdefghijklmnopqrstuvwxyz') for _ in range(len(english)))
print(f'english IoC ≈ {ioc(english):.4f}   (≈0.066 for English)')
print(f'random  IoC ≈ {ioc(noise):.4f}   (≈0.038 uniform)')
print(f'marked toy stream IoC ≈ {ioc(" ".join(generate(200, key=KEY))):.4f}  — the watermark is invisible here')


## 5 · Where to go from here

- **The real thing**: [`google-deepmind/synthid-text`](https://github.com/google-deepmind/synthid-text) —
  the open-sourced Nature-2024 implementation of the sampling-source scheme.
- **The papers**: Aaronson 2022 (the keyed-randomness proposal), Kirchenbauer et al. 2023
  (arXiv:2301.10226, green lists), Dathathri et al. 2024 (Nature, SynthID-Text) — all hosted on
  the ilm.red AI club alongside the post.
- **What this lab deliberately does not do**: build or evaluate watermark *removal*. Robustness is
  studied in the published literature; this notebook stays on the detection side of that line.
